# Obstacle recognition v4 (YOLO11n and YOLO11s, Kaggle)

Classes: **person, pallet, cart, forklift** (pallet jacks, pallet trucks, reach trucks, stackers and order pickers count as forklifts; roll cages count as carts).

What changed from v3 (see `AI_REVIEW.md`):
- `obstacles_v4`: 12,000 Gazebo images, now with built platform carts (thin handles), roll cages, shelf trolleys, reach trucks, stackers, order pickers and loads on forks; objects too thin to box are greyed out.
- Real photos: `loco_yolo_v4` (LOCO, thin boxes greyed out, people found by YOLO11m + YOLO-World-L), `coco_people` (COCO people, open licences only) and the checked Roboflow sets `rf_*` (CC BY 4.0).
- **Repeat-factor sampling**: images with rare classes (forklift, cart, person) are shown up to 3 times per epoch.
- Two model sizes: run the notebook once with `MODEL = 'yolo11n.pt'` and once with `'yolo11s.pt'` (cell 3). The larger one is kept only if it scores better.

Held-out tests (never trained on): `obstacles_v4_test` (new), `obstacles_v3_test` and `obstacles_v1_test` (comparison), and the **real LOCO test** (two other sites), in its v4 form and the v3 form v3 was scored on.

**Settings** (right panel): Accelerator **GPU T4 x2** (one is used), Internet **On**, **Persistence: Files only** (so a stopped session can resume). **Add Input**: your datasets `tugger-obstacles-v4` and `tugger-obstacles-v3` (the v3 one gives the old tests).

Run the cells in order. YOLO11n takes about 5-6 hours, YOLO11s about 9-10. If the session stops: run cells 1, 2, 3, then cell 5 (resume).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.3.40

In [ ]:
# Find every uploaded dataset (a folder with images/ and labels/), copy it to /content
# (32 files at a time: Kaggle input is slow per file), write the yaml files and the training
# list with repeat-factor sampling.
import glob, os, shutil, zipfile, yaml, time, math, random
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
IN, TMP, DATA = '/kaggle/input', '/kaggle/temp_unzip', '/content'
for z in glob.glob(f'{IN}/**/*.zip', recursive=True):
    zipfile.ZipFile(z).extractall(TMP)

def copy_one(pair):
    s, d = pair
    if not (os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s)):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        shutil.copyfile(s, d)

found = {}
for root in (IN, TMP):
    for d, subdirs, _ in os.walk(root):
        if 'images' in subdirs and 'labels' in subdirs and os.path.basename(d) not in found:
            found[os.path.basename(d)] = d
for name, src in sorted(found.items()):
    dst = f'{DATA}/{name}'
    pairs = [(os.path.join(d, f), os.path.join(dst, os.path.relpath(os.path.join(d, f), src)))
             for d, _, files in os.walk(src) for f in files]
    t = time.time()
    with ThreadPoolExecutor(32) as pool:
        list(pool.map(copy_one, pairs))
    print(f'{name}: {len(pairs)} files in {time.time() - t:.0f} s')

TRAIN_SETS = ['obstacles_v4', 'loco_yolo_v4', 'coco_people'] + sorted(n for n in found if n.startswith('rf_'))
names = {0: 'person', 1: 'pallet', 2: 'cart', 3: 'forklift'}

# Repeat-factor sampling (LVIS): r(c) = max(1, sqrt(T / f(c))), f(c) = share of training images
# with class c; an image is repeated by the largest r of its classes, at most MAX_REPEAT times.
T, MAX_REPEAT = 0.25, 3
images = []
for s in TRAIN_SETS:
    for p in sorted(glob.glob(f'{DATA}/{s}/images/train/*')):
        lab = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        cls = {int(l.split()[0]) for l in open(lab).read().split('\n') if l.strip()} if os.path.exists(lab) else set()
        images.append((p, cls))
freq = Counter(c for _, cls in images for c in cls)
r_c = {c: max(1.0, math.sqrt(T / (freq[c] / len(images)))) for c in names}
print('images with class:', {names[c]: freq[c] for c in names}, 'of', len(images))
print('class repeat factor:', {names[c]: round(r_c[c], 2) for c in names})
rng = random.Random(7)
listed = []
for p, cls in images:
    r = min(MAX_REPEAT, max([r_c[c] for c in cls], default=1.0))
    k = int(r) + (rng.random() < r - int(r))
    listed += [p] * k
open(f'{DATA}/train_v4.txt', 'w').write('\n'.join(listed))
boxes = Counter()
for p in listed:
    lab = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    boxes.update(int(l.split()[0]) for l in open(lab).read().split('\n') if l.strip())
print(f'training list: {len(listed)} entries; boxes per class (with repeats):', {names[c]: boxes[c] for c in names})

val = [f'{s}/images/val' for s in TRAIN_SETS if os.path.isdir(f'{DATA}/{s}/images/val')]
yaml.safe_dump({'path': DATA, 'names': names, 'train': f'{DATA}/train_v4.txt', 'val': val},
               open(f'{DATA}/train_v4.yaml', 'w'))
TESTS = {'v4_test': 'obstacles_v4_test/images/test', 'v3_test': 'obstacles_v3_test/images/test',
         'v1_test': 'obstacles_v1_test/images/test', 'loco_real_test_v4': 'loco_yolo_v4/images/test',
         'loco_real_test_v3form': 'loco_yolo/images/test'}
TESTS = {k: v for k, v in TESTS.items() if os.path.isdir(f'{DATA}/{v}')}
for test, folder in TESTS.items():
    yaml.safe_dump({'path': DATA, 'names': names, 'train': folder, 'val': folder}, open(f'{DATA}/{test}.yaml', 'w'))
    print(test, len(os.listdir(f'{DATA}/{folder}')), 'images')
print('val:', {v: len(os.listdir(f'{DATA}/{v}')) for v in val})

In [ ]:
# Shared set-up (Kaggle's Ray version breaks Ultralytics' Ray Tune link). Choose the model size here.
MODEL = 'yolo11n.pt'            # first run: 'yolo11n.pt'; second run: 'yolo11s.pt'
import numpy as np
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid
from ultralytics import YOLO
import ultralytics.utils.callbacks.raytune as raytune
raytune.callbacks.clear()
RUNS = '/kaggle/working/runs'
NAME = 'obstacles_v4_' + MODEL.split('.')[0].replace('yolo11', '')
print(NAME)

In [ ]:
# Train from the COCO weights. Same settings as v1-v3 (data and model size changed), early stop after 20 epochs without gain.
model = YOLO(MODEL)
model.train(data='/content/train_v4.yaml', imgsz=640, epochs=100, batch=32,
            patience=20, fliplr=0.5, flipud=0.0, degrees=0.0, mosaic=1.0, scale=0.5,
            translate=0.1, hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, weight_decay=0.0005,
            project=RUNS, name=NAME, seed=11, deterministic=True)

In [ ]:
# ONLY if the session stopped: run cells 2, 3, 4 and this one (not cell 5); it continues from the
# last finished epoch. Check first that /kaggle/working/runs/<NAME>/weights/last.pt exists.
YOLO(f'{RUNS}/{NAME}/weights/last.pt').train(resume=True)

In [ ]:
# Overfitting check: training and validation losses side by side.
import pandas as pd, matplotlib.pyplot as plt
r = pd.read_csv(f'{RUNS}/{NAME}/results.csv')
r.columns = [c.strip() for c in r.columns]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, name in zip(axes, ['box_loss', 'cls_loss']):
    ax.plot(r['epoch'], r[f'train/{name}'], label='train')
    ax.plot(r['epoch'], r[f'val/{name}'], label='validation')
    ax.set_title(name); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RUNS}/{NAME}/overfitting_check.png', dpi=150); plt.show()

In [ ]:
# Scores: mAP on validation and every held-out test, and the safety number: share of objects
# found by size (box height in px; > 200 px is roughly within 2-3 m), confidence 0.25, IoU 0.5.
import json, glob
best = YOLO(f'{RUNS}/{NAME}/weights/best.pt')
summary = {}

def recall_by_size(folder, bands=((0, 50), (50, 100), (100, 200), (200, 10000))):
    found, total = Counter(), Counter()
    for p in sorted(glob.glob(f'{DATA}/{folder}/*')):
        r = best.predict(p, conf=0.25, verbose=False)[0]
        h, w = r.orig_shape
        pb, pc = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)
        lab = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        for l in open(lab).read().split('\n'):
            if not l.strip():
                continue
            c, x, y, bw, bh = (float(v) for v in l.split())
            g = np.array([(x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h])
            band = next(i for i, (lo, hi) in enumerate(bands) if lo <= bh * h < hi)
            total[(int(c), band)] += 1
            if len(pb):
                ix = np.clip(np.minimum(pb[:, 2], g[2]) - np.maximum(pb[:, 0], g[0]), 0, None) * \
                     np.clip(np.minimum(pb[:, 3], g[3]) - np.maximum(pb[:, 1], g[1]), 0, None)
                iou = ix / ((pb[:, 2] - pb[:, 0]) * (pb[:, 3] - pb[:, 1]) + (g[2] - g[0]) * (g[3] - g[1]) - ix)
                found[(int(c), band)] += bool(((iou >= 0.5) & (pc == int(c))).any())
    return {names[c]: {f'{lo}-{hi if hi < 10000 else "max"} px': (round(found[(c, i)] / total[(c, i)], 3) if total[(c, i)] else None, total[(c, i)])
                       for i, (lo, hi) in enumerate(bands)} for c in names if any(total[(c, i)] for i in range(len(bands)))}

for name, data in [('val', '/content/train_v4.yaml')] + [(t, f'/content/{t}.yaml') for t in TESTS]:
    r = best.val(data=data, imgsz=640, project=RUNS, name=f'{NAME}_{name}')
    summary[name] = {'mAP50-95': float(r.box.map), 'mAP50': float(r.box.map50),
                     'per_class_mAP50-95': {r.names[c]: float(m) for c, m in zip(r.box.ap_class_index, r.box.maps[r.box.ap_class_index])}}
for t in ('loco_real_test_v4', 'v4_test'):
    if t in TESTS:
        summary[t]['found_by_size'] = recall_by_size(TESTS[t])
print(json.dumps(summary, indent=1))
json.dump(summary, open(f'{RUNS}/{NAME}/summary.json', 'w'), indent=1)

In [ ]:
!cd {RUNS} && zip -q -r /kaggle/working/{NAME}.zip {NAME}* -x '*/weights/last.pt'
from IPython.display import FileLink
FileLink(f'{NAME}.zip')   # click to download (also under Output on the right)